In [0]:
### This is an end-to-end coding playbook that covers real-time industry specific coding questions based on certains scenarios

In [0]:
### Initializing the spark session 
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import*
spark=SparkSession.builder.appName("coding").getOrCreate()

In [0]:
#### QUESTION 2

### PROBLEM STATEMENT: Meridian is a 22-year-old IT services consultancy. The HR analytics team supports compensation reviews, attrition reporting, and the annual bonus calculation that goes to 8,500 employees. Last quarter's salary review surfaced inconsistencies. People were getting different ranks across reports for the same data. The Head of HR wants one source of truth and one set of definitions before the next cycle.

#### DATA COLLECTION: The Head of HR wants definitive employee rankings and bonus calculations. The CFO wants the rules expressed clearly enough that anyone can re-run them. The data team needs to demonstrate the right way to do rankings on the same dataset.

employee_df= spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/2_meridianconsulting/raw_data/employees.csv")

departments_df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/2_meridianconsulting/raw_data/departments.csv")

employee_salary_df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/2_meridianconsulting/raw_data/employee_salary.csv")

bonus_policy_df= spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/2_meridianconsulting/raw_data/bonus_policy.csv")

grades_df= spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/2_meridianconsulting/raw_data/grades.csv")

performance_df= spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/2_meridianconsulting/raw_data/performance_reviews.csv")

In [0]:
### Task 1: Deliverable 1 – Data Quality Validation

### Before performing any analytical transformations or generating business reports, the HR datasets must undergo a comprehensive data quality validation process to ensure the accuracy and reliability of downstream calculations. 

# Analyze the employee, salary, and performance review datasets to identify records that violate business or data integrity rules. The validation process should detect duplicate employee IDs, duplicate email addresses, employees reporting to themselves, invalid manager IDs that do not exist in the employee master, future joining dates, null salaries, negative or zero salary values, null performance scores, and duplicate performance review records for the same employee and review year. 

# Generate separate reports for each type of data quality issue so that the HR Operations team can review and rectify the anomalies before the data is used for employee rankings, salary analysis, or bonus calculations. These validation reports will serve as an essential data quality checkpoint to ensure that all subsequent analyses are performed on clean, consistent, and trustworthy data.

In [0]:
### Performing the data cleaning and data transformation steps here for the EMPLOYEE DATASET

### Segregating the valid and non valid records first
valid_employee_df= employee_df.filter((f.col("employee_id").isNotNull())&(f.col("joining_date").isNotNull())&(f.col("gender").isNotNull())&((f.col("status")!=f.lit("Notice Period"))&(f.col("status")!=f.lit("Terminated")))&(f.col("email").isNotNull()))

non_valid_employee_df=employee_df.filter((f.col("employee_id").isNull())|(f.col("joining_date").isNull())|(f.col("email").isNull())|(f.col("gender").isNull()))

### Now that we have found our valid employee records now we will have to perform the prelimiary data cleaning
valid_employee_df= valid_employee_df.withColumn("employee_name",f.trim(f.initcap(f.col("employee_name"))))
valid_employee_df= valid_employee_df.withColumn("gender",f.trim(f.initcap(f.col("gender"))))
valid_employee_df= valid_employee_df.withColumn("email",f.trim(f.initcap(f.col("email"))))
valid_employee_df= valid_employee_df.withColumn("joining_date",f.when(f.col("joining_date").cast("date").isNull(),f.lit("NULL")).otherwise(f.col("joining_date").cast("date")))
valid_employee_df= valid_employee_df.withColumn("designation",f.trim(f.initcap(f.col("designation"))))
valid_employee_df= valid_employee_df.withColumn("employment_type",f.trim(f.initcap(f.col("employment_type"))))
valid_employee_df= valid_employee_df.withColumn("status",f.trim(f.initcap(f.col("status"))))
valid_employee_df=valid_employee_df.filter(f.col("gender")!=f.lit("X")) 

#### The next step is to find the latest record pertaining to each of the employees pertainign to their joining date columns fo which we will be using the window functions

from pyspark.sql.window import Window
window_created= Window.partitionBy([f.col("employee_name")]).orderBy(f.col("joining_date").desc())
valid_employee_df= valid_employee_df.withColumn("rn",f.row_number().over(window_created))
valid_employee_df= valid_employee_df.filter(f.col("rn")==1).drop(f.col("rn"))

valid_employee_df.show(5)

#### After we are done with the data cleaning of the dataset we will push it to some other volume where only the valid datasets will be present. For the time beign we are usign the mode OVERWRITE because this is just a problem solving daatset

valid_employee_df.write.format("delta").mode("overwrite").save("/Volumes/coding_playbook/2_meridianconsulting/final_data/employee_final")

In [0]:
### Performing the data cleaning and data transformation steps here for the DEPARTMENT DATASET

valid_departments_df=departments_df.withColumn("annual_budget",f.col("annual_budget").cast("int"))
valid_departments_df=valid_departments_df.withColumn("department_name",f.initcap(f.trim(f.col("department_name"))))
valid_departments_df=valid_departments_df.withColumn("location",f.initcap(f.trim(f.col("location"))))
valid_departments_df.show()

### After performing the data cleaning we will push it back to the final volume where only the valid data is lying.
departments_df.write.format("delta").mode("overwrite").save("/Volumes/coding_playbook/2_meridianconsulting/final_data/departments_final")

In [0]:
### Performing the data cleaning and data transformation steps here for the EMPLOYEE SALARY DATASET

valid_employee_salary_df= employee_salary_df.withColumn("salary",f.abs(f.col("salary").cast("int")))
valid_employee_salary_df= valid_employee_salary_df.withColumn("effective_date",f.col("effective_date").cast("date"))
valid_employee_salary_df= valid_employee_salary_df.withColumn("currency",f.upper(f.trim(f.col("currency"))))
valid_employee_salary_df.show()

### After performing the data cleaning we will push it back to the final volume where only the valid data is lying.
valid_employee_salary_df.write.format("delta").mode("overwrite").save("/Volumes/coding_playbook/2_meridianconsulting/final_data/employee_salary_final")

In [0]:
### Performing the data cleaning and data transformation steps here for the GRADES DATASET

valid_grades_df= grades_df.withColumn("min_salary",f.abs(f.col("min_salary").cast("int")))
valid_grades_df= valid_grades_df.withColumn("max_salary",f.col("max_salary").cast("int"))
valid_grades_df= valid_grades_df.withColumn("grade_name",f.upper(f.trim(f.col("grade_name"))))
valid_grades_df.show()

### After performing the data cleaning we will push it back to the final volume where only the valid data is lying.
valid_grades_df.write.format("delta").mode("overwrite").save("/Volumes/coding_playbook/2_meridianconsulting/final_data/grades_final")

In [0]:
### Performing the data cleaning and data transformation steps here for the PERFORMANCES DATASET

valid_performance_df= performance_df.withColumn("performance_score",f.col("performance_score").cast("int"))
valid_performance_df= valid_performance_df.withColumn("reviewer",f.upper(f.trim(f.col("reviewer")))).fillna(0)
valid_performance_df.show()

### After performing the data cleaning we will push it back to the final volume where only the valid data is lying.
valid_performance_df.write.format("delta").mode("overwrite").save("/Volumes/coding_playbook/2_meridianconsulting/final_data/performance_final")

In [0]:
## TASK 2: Deliverable 2 – Employee Hierarchy
### Using the employee master dataset, establish the reporting hierarchy by retrieving each employee's manager information through a self-join. Employees without managers should still appear in the final output.

employee_df_final= spark.read.format("delta").load("/Volumes/coding_playbook/2_meridianconsulting/final_data/employee_final/")

emp = (employee_df_final.select(f.col("employee_id"), "employee_name", f.col("manager_id")).alias("a"))

mgr = (employee_df_final.select(f.col("employee_id"),f.col("employee_name").alias("manager_name")).alias("b"))

employee_manager_hierarchy = (emp.join(mgr,f.col("a.manager_id") == f.col("b.employee_id"),"inner")).drop(f.col("b.employee_id")).withColumn("manager_id",f.col("manager_id").cast("string"))

employee_manager_hierarchy.show()

In [0]:
employee_df_final.show(5)

In [0]:
## TASK 3: Deliverable 3 – Salary Ranking within Grade
### Determine each employee's salary standing within their respective department using the following ranking techniques: ROW_NUMBER(), RANK(), DENSE_RANK(). Aftwewards, Compare the differences in rankings where multiple employees earn identical salaries.

req_emp_info= employee_df_final.select(f.col("employee_id"),f.col("employee_name"),f.col("employment_type"),f.col("department_id"))
req_salary_df= spark.read.format("delta").load("/Volumes/coding_playbook/2_meridianconsulting/final_data/employee_salary_final/").select(f.col("employee_id"),f.col("salary"))

emp_salary_df= req_emp_info.alias("a").join(req_salary_df.alias("b"),on=f.col("a.employee_id").cast("string")==f.col("b.employee_id").cast("string")).select(f.col("a.employee_id"),f.col("a.employee_name"),f.col("a.employment_type"),f.col("department_id"),f.col("b.salary").cast("int"))

#### Uisng the row functions we will create a window function that will help us fetch the descending orders of the salaries
from pyspark.sql.window import Window
window_created_1= Window.partitionBy(f.col("department_id")).orderBy(f.col("salary").desc())

emp_salary_df= emp_salary_df.withColumn("dept_salary_rn",f.rank().over(window_created_1))
emp_salary_df.show()


In [0]:
## TASK 4: Deliverable 4 – Top Three Performers by Department
### Identify the top three performing employees in every department based on their latest performance review. In situations where employees have identical performance scores, compare the behavior of different ranking functions.

req_emp_info= employee_df_final.select(f.col("employee_id"),f.col("employee_name"),f.col("employment_type"),f.col("department_id"))
performance_df_final= spark.read.format("delta").load("/Volumes/coding_playbook/2_meridianconsulting/final_data/performance_final/")

max_review_year= performance_df.select(f.col("review_year")).orderBy(f.col("review_year").desc()).limit(1).collect()[0][0]
performance_df_req= performance_df_final.filter(f.col("review_year")==max_review_year)

### Now comes the final join part and then finding the top 3 employees every department based on their performance review
emp_performance= req_emp_info.alias("a").join(performance_df_req.alias("b"),on=f.col("a.employee_id").cast("string")==f.col("b.employee_id").cast("string")).select("a.*",f.col("b.performance_score"))

from pyspark.sql.window import Window
window_created_2= Window.partitionBy(f.col("department_id")).orderBy(f.col("performance_score").cast("int").desc())

emp_performance= emp_performance.withColumn("performance_rank",f.rank().over(window_created_2))
final_emp_performance= emp_performance.filter(f.col("performance_rank")<=3)
final_emp_performance.show(5)

In [0]:
final_emp_performance.filter((f.col("department_id")==1)&(f.col("performance_rank")==2)).show()

In [0]:
max_review_year= performance_df.select(f.col("review_year")).orderBy(f.col("review_year").desc()).limit(1).collect()[0][0]
max_review_year

In [0]:
req_emp_info.show(5)
performance_df.show(5)

In [0]:
### TASK 5: Deliverable 5 – Departments Requiring Salary Review
## Generate departmental salary statistics and identify departments meeting the following conditions:
# Average Salary greater than ₹120,000
# Employee Count greater than 25
# Total Salary Expense greater than ₹5,000,000
## Implement the filtering using aggregate functions and the HAVING clause.

department_statistics= emp_salary_df.groupBy(f.col("department_id")).agg(f.count("*").alias("total_active_employees"),
                                                                         f.round(f.mean(f.col("salary")),3).alias("average_salary"),f.round(f.sum("salary"),3).alias("total_salary_spend"))

department_statistics_final= department_statistics.filter((f.col("total_active_employees")>25)&(f.col("average_salary")>120000)&(f.col("total_salary_spend")>5000000)).orderBy(f.col("department_id").asc())

department_statistics_final.show()

In [0]:
### TASK 6: Deliverable 6 – Employee Bonus Calculation

## As part of the annual compensation review process, the HR department requires a standardized bonus calculation to ensure consistency across the organization. Using the latest salary and performance review data, calculate the bonus amount for every employee based on the predefined business rules. 

# Employees with a performance score of **95 or above** and a salary below ₹70,000 are eligible for a 25% bonus. 

# Employees with a performance score of **90 to 94** are eligible for a **20% bonus**, those scoring **80 to 89** receive a 15% bonus, employees scoring 70 to 79 receive a 10% bonus, and employees scoring below 70 receive a 5% bonus. 

# However, employees whose employment type is Contrac* or Intern, as well as employees whose employment status is Terminated, are not eligible for any bonus and should receive a bonus percentage of 0%..

employee_performance_df_final=(emp_salary_df.select(f.col("employee_id"),f.col("employee_name"),f.col("salary"),f.col("employment_type"))).alias("a").join((valid_performance_df.select(f.col("employee_id"),f.col("performance_score"),f.col("review_year"))).alias("b"),on=f.col("a.employee_id")==f.col("b.employee_id"),how="left").drop(f.col("b.employee_id")).filter(f.col("review_year")==2025)

employee_performance_df_final=employee_performance_df_final.filter(~f.col("employment_type").isin("Contract","Terminated","Intern"))

performance_bonus_df= employee_performance_df_final.withColumn("performance_score_1",f.when((f.col("performance_score").cast("int")>=95)&(f.col("salary")<70000),f.col("salary")+f.col("salary")*0.25))

performance_bonus_df= performance_bonus_df.withColumn("performance_score_2",f.when((f.col("performance_score").cast("int")>=90)&(f.col("performance_score").cast("int")<=94),f.col("salary")+f.col("salary")*0.20))

performance_bonus_df= performance_bonus_df.withColumn("performance_score_3",f.when((f.col("performance_score").cast("int")>=80)&(f.col("performance_score").cast("int")<90),f.col("salary")+f.col("salary")*0.15))

performance_bonus_df= performance_bonus_df.withColumn("performance_score_4",f.when((f.col("performance_score").cast("int")>=70)&(f.col("performance_score").cast("int")<79),f.col("salary")+f.col("salary")*0.10))

performance_bonus_df=performance_bonus_df.withColumn("revised_salary",f.coalesce(f.col("performance_score_1"),f.col("performance_score_2"),f.col("performance_score_3"),f.col("performance_score_4")))

performance_bonus_df=performance_bonus_df.withColumn("final_revised_salary",f.coalesce(f.col("revised_salary"),f.col("salary")))

performance_bonus_df=performance_bonus_df.drop(f.col("revised_salary"),f.col("performance_score_1"),f.col("performance_score_2"),f.col("performance_score_3"),f.col("performance_score_4"))

performance_bonus_df.show()